# Reproducing Section 6

**De-Anonymizing and $k$-Anonymizing Individuals using Subject-Object Knowledge Relations**

This notebook is a *driver*. Every algorithm lives in `harness.py` and nothing is
reimplemented here. That is deliberate: two copies of an algorithm drift apart, and the
numbers in the paper must come from exactly one place.

Run the cells top to bottom.

**Required in this folder:** `harness.py`, `test.xlsx`, `COSE.ipynb`

## 1. Setup

In [1]:
import importlib, importlib.util, json, os, subprocess, sys, shutil
import numpy as np
import pandas as pd

DATA = "test.xlsx"        # the Nextcloud export
OUT  = "results"          # where CSVs are written
NB   = "COSE.ipynb"       # original prototype, needed only for the self-test

for f in (DATA, "harness.py"):
    print(("  found   " if os.path.exists(f) else "  MISSING ") + f)
print(("  found   " if os.path.exists(NB) else "  MISSING ") + NB + "   (self-test only)")

  found   test.xlsx
  found   harness.py
  found   COSE.ipynb   (self-test only)


In [2]:
# Import harness.py as a module. importlib.reload picks up edits without restarting
# the kernel, which a plain `import harness` will not do.
import harness
importlib.reload(harness)
print("harness.py loaded from", os.path.abspath(harness.__file__))

harness.py loaded from C:\Users\Shahzad\Secure Systems\Whistleblowing and k-Anonymity\NEW\harness.py


## 2. Build `mod.py` for the self-test

`mod.py` is the original notebook exported as an importable script. The self-test runs
`kAnonHomogenize` at $k=3$ through both the prototype and the harness on the same graph and
compares the edge counts. Without `mod.py` the check prints `SKIPPED` rather than passing
silently.

Skip this cell if `mod.py` already exists.

In [3]:
if os.path.exists("mod.py"):
    print("mod.py already present, skipping conversion")
elif os.path.exists(NB):
    r = subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "script", NB],
                       capture_output=True, text=True)
    produced = NB.replace(".ipynb", ".py")
    if os.path.exists(produced):
        shutil.move(produced, "mod.py")
        print("wrote mod.py")
    else:
        print("nbconvert failed:\n", r.stderr[-800:])
else:
    print(f"{NB} not found. The self-test will report SKIPPED.")

mod.py already present, skipping conversion


## 3. Run everything

In [4]:
# Passing an explicit argv list is required inside Jupyter: the kernel injects
# '-f <kernel.json>' into sys.argv, so harness.py ignores sys.argv in a notebook and uses
# the defaults unless you hand it arguments here.
args = harness.main(["--data", DATA, "--out", OUT, "--seeds", "30", "--selftest", "--surrogate"])

[run] data=C:\Users\Shahzad\Secure Systems\Whistleblowing and k-Anonymity\NEW\test.xlsx
[run] out=C:\Users\Shahzad\Secure Systems\Whistleblowing and k-Anonymity\NEW\results  seeds=30
dataset: {
  "subjects": 37,
  "objects": 1824,
  "edges": 5484,
  "mean_jaccard": 0.029118042993876304,
  "distinct_neighbourhoods": 31,
  "max_degree": 1251,
  "median_degree": 11,
  "min_degree": 1
}
[selftest] original=6726  harness=6726  match=True

[1/4] real dataset ...
                         method  k  edges_added        pct  nodes_added  runtime_s  utility_js  utility_legacy property_verified
                 DeAnonMinGraph              28   0.510576            0   0.026115    0.422786             0.0              True
                kAnonHomogenize  2         2728  49.744712            0   0.012389    0.237887             0.0              True
        kAnonHomogenize-Overlap  2         3019  55.051058            0   0.009616    0.212564             0.0              True
           k-Degree (Li

### Reading the self-test line

| Output | Meaning |
|---|---|
| `match=True` | The harness reproduces the prototype. Proceed. |
| `match=False` | **Stop.** The rewrite is not behaviour-preserving and the results are not trustworthy. |
| `SKIPPED` | `mod.py` is missing. Equivalence was **not** checked, so do not cite the self-test. |

The check covers `kAnonHomogenize` at $k=3$. It does not cover `k_degree_anonymize` or
`js_utility`, which are new rather than ported, and so have no prototype counterpart.

## 4. The tables as they appear in the paper

In [5]:
meta = json.load(open(f"{OUT}/dataset_meta.json"))
print(f"Dataset: {meta['subjects']} subjects, {meta['objects']} objects, "
      f"{meta['edges']} edges, mean Jaccard {meta['mean_jaccard']:.4f}")
print(f"Degrees: min {meta['min_degree']}, median {meta['median_degree']}, max {meta['max_degree']}")
print(f"Distinct neighbourhoods: {meta['distinct_neighbourhoods']} of {meta['subjects']}")

Dataset: 37 subjects, 1824 objects, 5484 edges, mean Jaccard 0.0291
Degrees: min 1, median 11, max 1251
Distinct neighbourhoods: 31 of 37


In [6]:
# Table 5 -- all methods on the real graph
real = pd.read_csv(f"{OUT}/real_results.csv")
t5 = real.copy()
t5["pct"] = t5["pct"].round(1)
t5["utility_js"] = t5["utility_js"].round(3)
t5["runtime_s"] = t5["runtime_s"].round(3)
t5[["method", "k", "edges_added", "pct", "utility_js", "runtime_s", "property_verified"]]

,method,k,edges_added,pct,utility_js,runtime_s,property_verified
0,DeAnonMinGraph,NaN,28,0.5,0.423,0.026,True
1,kAnonHomogenize,2.0,2728,49.7,0.238,0.012,True
2,kAnonHomogenize-Overlap,2.0,3019,55.1,0.213,0.010,True
3,k-Degree (Liu-Terzi),2.0,898,16.4,0.591,0.004,NaN
4,Lower bound (degree relaxation),2.0,898,16.4,NaN,0.000,NaN
5,kAnonHomogenize,3.0,6726,122.6,0.085,0.011,True
6,kAnonHomogenize-Overlap,3.0,6399,116.7,0.113,0.007,True
7,k-Degree (Liu-Terzi),3.0,1700,31.0,0.438,0.002,NaN
8,Lower bound (degree relaxation),3.0,1700,31.0,NaN,0.000,NaN
9,kAnonHomogenize,5.0,10385,189.4,0.054,0.006,True


In [7]:
# Table 6 -- cost against measured neighbourhood overlap, with 95% bootstrap intervals
ov = pd.read_csv(f"{OUT}/overlap_sweep.csv")
rows = []
for rf, s in ov.groupby("role_frac"):
    m, lo, hi = harness.boot_ci(s.kanon_overlap_pct)
    rows.append(dict(jaccard=round(s.jaccard.mean(), 2),
                     kAnonHomogenize=round(s.kanon_pct.mean(), 1),
                     overlap_aware=round(m, 1),
                     ci=f"[{lo:.1f}, {hi:.1f}]",
                     k_degree=round(s.kdegree_pct.mean(), 1),
                     deanon=round(s.deanon_pct.mean(), 2)))
t6 = pd.DataFrame(rows)
t6

,jaccard,kAnonHomogenize,overlap_aware,ci,k_degree,deanon
0,0.05,176.5,167.1,"[165.8, 168.4]",3.8,0.00
1,0.06,173.0,159.3,"[157.9, 160.7]",3.7,0.00
2,0.07,169.6,148.2,"[146.6, 149.8]",3.8,0.00
3,0.08,168.3,135.4,"[133.7, 137.1]",3.9,0.00
4,0.09,166.9,117.4,"[114.9, 120.0]",3.9,0.00
5,0.10,164.7,98.9,"[96.7, 101.0]",3.7,0.01
6,0.11,165.8,81.7,"[78.8, 84.8]",3.8,0.03
7,0.12,165.1,71.9,"[68.3, 75.5]",3.8,1.39


In [8]:
# Table 7 -- scalability, both algorithms on one common ensemble
sc = pd.read_csv(f"{OUT}/scalability.csv")
t7 = sc.groupby("n_subjects")[["base_edges", "kanon_pct", "kanon_time",
                               "deanon_pct", "deanon_time"]].mean().round(3)
t7

,base_edges,kanon_pct,kanon_time,deanon_pct,deanon_time
n_subjects,,,,,
50,3099.2,175.851,0.009,0.000,0.002
100,6268.8,165.270,0.029,0.000,0.003
200,12602.9,165.412,0.092,0.001,0.007
400,25241.3,164.012,0.403,0.001,0.031
800,49658.7,163.591,1.961,0.005,0.174
1600,100075.3,163.307,8.917,0.007,1.269


In [9]:
# Section 6.6 -- sensitivity to subject processing order
od = pd.read_csv(f"{OUT}/ordering_sensitivity.csv")
for col in ("kAnonHomogenize", "kAnonHomogenize_Overlap"):
    m, lo, hi = harness.boot_ci(od[col])
    print(f"{col:26s} mean {m:9.1f}   95% CI [{lo:.1f}, {hi:.1f}]   "
          f"min {od[col].min()}  max {od[col].max()}")

kAnonHomogenize            mean   10111.8   95% CI [9554.4, 10648.9]   min 6858  max 13138
kAnonHomogenize_Overlap    mean    6399.0   95% CI [6399.0, 6399.0]   min 6399  max 6399


## 5. Consistency check

Verifies that the figures quoted in the manuscript still match the CSVs just produced. Run
this before every submission. The three transcription errors that this project has already
hit were all of the same kind: a number produced somewhere other than the pipeline of
record, then copied into the text by hand.

In [10]:
TEX = "DeAnon_KAnon.tex"   # set to your manuscript path, or leave and skip if absent

def claim(label, cond):
    print(("  ok    " if cond else "  FAIL  ") + label)
    return bool(cond)

def row(method, k=None):
    r = real[real.method == method]
    r = r[r.k.isna()] if k is None else r[r.k == k]
    return r.iloc[0]

results = []
results.append(claim("dataset 37 / 1824 / 5484",
                     (meta["subjects"], meta["objects"], meta["edges"]) == (37, 1824, 5484)))
results.append(claim("mean Jaccard 0.029", round(meta["mean_jaccard"], 3) == 0.029))
results.append(claim("DeAnonMinGraph 28 edges (0.5%)",
                     int(row("DeAnonMinGraph").edges_added) == 28))
results.append(claim("kAnonHomogenize k=3: 6726 edges, 122.6%, utility 0.085",
                     int(row("kAnonHomogenize", 3).edges_added) == 6726
                     and round(row("kAnonHomogenize", 3).pct, 1) == 122.6
                     and round(row("kAnonHomogenize", 3).utility_js, 3) == 0.085))
results.append(claim("k-Degree k=3: 1700 edges, 31.0%, utility 0.438",
                     int(row("k-Degree (Liu-Terzi)", 3).edges_added) == 1700
                     and round(row("k-Degree (Liu-Terzi)", 3).pct, 1) == 31.0
                     and round(row("k-Degree (Liu-Terzi)", 3).utility_js, 3) == 0.438))
results.append(claim("k-degree cost is monotone non-decreasing in k",
                     list(real[real.method == "k-Degree (Liu-Terzi)"].sort_values("k").edges_added)
                     == sorted(real[real.method == "k-Degree (Liu-Terzi)"].edges_added)))
results.append(claim("k-anonymity actually achieved on every kAnon run",
                     bool(real[real.method.str.startswith("kAnon")].property_verified.all())))
results.append(claim("overlap-aware variant is order-invariant",
                     od.kAnonHomogenize_Overlap.nunique() == 1))
results.append(claim("overlap sweep runs 167.1% down to 71.9%",
                     round(t6.overlap_aware.iloc[0], 1) == 167.1
                     and round(t6.overlap_aware.iloc[-1], 1) == 71.9))

print(f"\n{sum(results)}/{len(results)} checks passed")

  ok    dataset 37 / 1824 / 5484
  ok    mean Jaccard 0.029
  ok    DeAnonMinGraph 28 edges (0.5%)
  ok    kAnonHomogenize k=3: 6726 edges, 122.6%, utility 0.085
  ok    k-Degree k=3: 1700 edges, 31.0%, utility 0.438
  ok    k-degree cost is monotone non-decreasing in k
  ok    k-anonymity actually achieved on every kAnon run
  ok    overlap-aware variant is order-invariant
  ok    overlap sweep runs 167.1% down to 71.9%

9/9 checks passed


## 6. The surrogate dataset

`--surrogate` writes `results/surrogate_graph.csv`, a synthetic stand-in that matches the
source graph's subject count, object count, **exact degree sequence** and mean pairwise
Jaccard similarity.

It does **not** reproduce Table 5. At $k=3$ the surrogate needs 140.8% additional edges
against the real graph's 122.6%; at $k=2$, 89.9% against 49.7%. Mean overlap predicts cost
well within a graph family but does not determine it across families, because higher-order
structure such as the nesting of small neighbourhoods inside large ones also matters. This
is discussed in Section 6.9.

The surrogate is what gets published. **`test.xlsx` is not released**: it contains personal
data of identifiable individuals in both the account names and the folder paths, so
pseudonymising the account column would not make it safe.

In [12]:
if os.path.exists(f"{OUT}/surrogate_meta.json"):
    sm = json.load(open(f"{OUT}/surrogate_meta.json"))
    print(f"surrogate: {sm['subjects']} subjects, {sm['edges']} edges, J {sm['mean_jaccard']:.4f}")
    print(f"source:    {sm['source_subjects']} subjects, {sm['source_edges']} edges, "
          f"J {sm['source_mean_jaccard']:.4f}")
else:
    print("no surrogate written; add --surrogate to the call in section 3")

surrogate: 37 subjects, 5484 edges, J 0.0290
source:    37 subjects, 5484 edges, J 0.0291
